# Interpretative Jury Learning

Adjust **`RunConfig`** below, run **`run_full_pipeline(cfg)`**, then read metrics from **`result`**:

- **Training / validation (per epoch):** `result.training_metrics_df()` — columns include `train_loss`, `train_accuracy`, `val_loss`, `val_accuracy`, `learning_rate`.
- **Last epoch only:** `result.final_training_metrics()` — dict with the same fields.
- **Held-out test splits:** `result.eval_metrics_df()` or `result.eval_accuracy_by_split` — accuracies are fractions in `[0, 1]` (`validation` is the same val set used during training).

Set `cfg.verbose = False` for quiet runs; `cfg.show_progress_bar = True` for tqdm over batches.

**Optional plots** (continent breakdown, calibration, etc.) live in `jury_learning.plots`; see the section at the bottom.


In [ ]:
from jury_learning import RunConfig, run_full_pipeline

cfg = RunConfig()

cfg.db_path = "moral_machine.db"
cfg.model_path = "moral_jury_dcn_model.pth"

cfg.sql_subset_size = 100000
cfg.batch_size = 1024
cfg.eval_batch_size = 64

cfg.embed_dim = 128
cfg.hidden_dim = 512
cfg.num_cross_layers = 3
cfg.response_encoder_hidden = 64

cfg.epochs = 50
cfg.lr = 1e-3
cfg.lr_phase2 = 1e-4
cfg.freeze_encoder_epoch_fraction = 2 / 3

cfg.verbose = True
cfg.show_progress_bar = False
cfg.use_wandb = False

cfg.device = "auto"

cfg.run_training_stage = True
cfg.run_evaluation_stage = True

result = run_full_pipeline(cfg)


### Metrics


In [ ]:
# Per-epoch train + validation (same val loader as during training)
result.training_metrics_df()


In [ ]:
# Final epoch summary
result.final_training_metrics()


In [ ]:
# Accuracy on validation + stress-test splits (fractions 0–1)
result.eval_metrics_df()


### Optional evaluation plots

Requires **`pip install matplotlib`**; continent aggregation needs **`pip install pycountry_convert`**.

- **`run_evaluation_plots`** — split bar chart, training curves, calibration diagram, continent + country bars (validation and new-user splits).
- Use **`accuracy_by_country`** + **`continent_metrics_table`** / **`country_metrics_table`** if you want tables without plotting.


In [ ]:
from jury_learning.plots import run_evaluation_plots

run_evaluation_plots(
    cfg,
    result.bundle,
    result.model,
    split_metrics=result.eval_accuracy_by_split,
    history=result.training_history,
)
